# Kansas Wheat Crop Yield Prediction – Capstone Project

- Author: Denish Trada
- Date: 27th Apr, 2025

## 9. Baseline Modeling Overview

After completing data cleaning, feature engineering, and exploratory data analysis (EDA) in the previous notebook,  
we now proceed to build **baseline regression models** to predict wheat yield.

The goal of baseline modeling is:

- Establish initial benchmark performance
- Evaluate simple models (Linear Regression, Decision Tree, Random Forest)
- Identify which features contribute meaningfully
- Set up the evaluation framework for future model improvements

### 9.1 Data Loading and Setup

In [14]:
# Basic packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [52]:
# Modelling tools
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [20]:
# Loading the cleaned dataset
final_merged = pd.read_csv('Data_Processed/final_merged_cleaned.csv')

In [22]:
# Quick preview
final_merged.head()

,Year,County,County ANSI,Yield,CV (%),Growing_Season_Precip_mm,Growing_Season_Temp_C,NAME,mean,elevation,...,SQ2,SQ3,SQ4,SQ5,SQ6,SQ7,soil_quality_score,slope_avg,temp_precip_interaction,lag_yield
0,2012,ALLEN,1,NaN,11.2,309.74,23.479518,Allen,0.536703,316.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.125,7272.545992,NaN
1,2012,ALLEN,1,NaN,12.9,309.74,23.479518,Allen,0.536703,316.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.125,7272.545992,NaN
2,2017,ALLEN,1,NaN,29.8,738.92,21.172937,Allen,0.647254,316.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.125,15645.106282,NaN
3,2017,ALLEN,1,NaN,20.2,738.92,21.172937,Allen,0.647254,316.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.125,15645.106282,NaN
4,2017,ALLEN,1,NaN,20.1,738.92,21.172937,Allen,0.647254,316.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.125,15645.106282,NaN


### 9.2 Feature and Target Definition

In [59]:
# Defining target variable
target = 'Yield'

In [61]:
# Defining features to use - Note: Excluding 'lag_yield' for now due to extensive missing values
features = [
    'Growing_Season_Temp_C',
    'Growing_Season_Precip_mm',
    'soil_quality_score',
    'slope_avg',
    'temp_precip_interaction' ]

In [63]:
# Creating feature matrix X and target vector y
X = final_merged[features]
y = final_merged[target]

**Explanation:**
- I selected core engineered features that showed meaningful relationships with Yield during EDA.
- The target variable is the wheat yield measured in bushels per acre.

### 9.3 Train-Test Split

In [74]:
# Dropping rows where target variable Yield is missing
final_merged = final_merged.dropna(subset=['Yield']).reset_index(drop=True)


In [76]:
# Redefining feature matrix and target
X = final_merged[features]
y = final_merged['Yield']

**Fixing Missing Target Values**

Before proceeding with model training,  
it is crucial to ensure that the target variable (`Yield`) does not contain missing values.

- Scikit-learn estimators do not accept NaN values in the target variable.
- Rows with missing Yield were dropped to maintain modeling integrity.

After this cleaning step, the feature matrix (`X`) and target vector (`y`) are ready for proper train-test splitting and model training.

---


In [78]:
# Splitting into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Checking shape
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")


Training set size: 40 samples
Testing set size: 10 samples


**Train-Test Split Summary**

- **Training set size:** 549 samples (80% of the data)
- **Testing set size:** 138 samples (20% of the data)

This ensures that model training will occur on a substantial portion of the dataset, while performance evaluation will be done on unseen data to simulate real-world prediction scenarios.

Random state = 42 ensures **reproducibility** — meaning every time the notebook is run, the split remains identical.

---


## Baseline Model 1: Linear Regression

**Why Linear Regression?**

Linear Regression is one of the simplest and most interpretable machine learning models.  
It assumes a linear relationship between input features and the target variable (wheat yield, in this case).

Building a Linear Regression baseline model allows us to:

- Quickly assess whether there is **any usable linear pattern** between features and yield
- Establish a **benchmark RMSE/MAE/R²** performance level
- Serve as a reference for judging whether more complex models (like Decision Trees or Random Forests) provide meaningful improvements

Although agriculture is complex and often non-linear, a linear model gives us a **valuable first view** into the predictive power of our features.

---

**Model Setup Summary**

- **Features Used:**  
  - Growing season average temperature
  - Growing season total precipitation
  - Soil quality score
  - Average slope
  - Temperature × Precipitation interaction

- **Target Variable:**  
  - Wheat Yield (bushels per acre)

- **Data Split:**  
  - 80% training set
  - 20% testing set (randomized, stratified where needed)

---

In [81]:
# Initializing model
lr_model = LinearRegression()

# Train on training data
lr_model.fit(X_train, y_train)

# Predict on test data
y_pred_lr = lr_model.predict(X_test)


In [83]:
# Evaluation Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Calculate metrics
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

# Print results
print(f"Linear Regression RMSE: {rmse_lr:.2f}")
print(f"Linear Regression MAE: {mae_lr:.2f}")
print(f"Linear Regression R²: {r2_lr:.2f}")

Linear Regression RMSE: 344.88
Linear Regression MAE: 297.35
Linear Regression R²: -0.78


After training and testing the Linear Regression model, the following performance metrics were obtained:

| Metric | Value |
|:-------|:------|
| RMSE (Root Mean Squared Error) | 344.88 |
| MAE (Mean Absolute Error) | 297.35 |
| R² Score (Coefficient of Determination) | -0.78 |

---

**a. Root Mean Squared Error (RMSE)**

- RMSE measures the **average magnitude of the error** between predicted and actual yields.
- An RMSE of 344.88 bushels per acre is **very high**, considering that typical wheat yields range between 0–1000 Bu/Acre.
- This indicates that, on average, the Linear Regression model’s predictions are off by a very large margin — **major prediction errors**.

---

**b. Mean Absolute Error (MAE)**

- MAE of 297.35 further confirms the large average errors.
- Since MAE is almost 300 Bu/Acre, the model is **missing nearly half or more of actual yield values** on average.
- This level of error is considered unacceptable for any practical agricultural forecasting use case.

---

**c. R² Score (Coefficient of Determination)**

- An R² score of **-0.78** indicates **very poor model fit**.
- Normally, R² ranges between 0 (no fit) and 1 (perfect fit).  
- A negative R² means the model performs **worse than simply predicting the mean** of the training data for all cases.
- In other words, the model is not learning any meaningful pattern from the features to predict yields.

---

**Conclusion:**

 The Linear Regression baseline model **did not perform well**:

- The relationship between features and wheat yield appears to be **highly non-linear** and/or **noisy**.
- A simple linear assumption is not sufficient to capture the complex, real-world agricultural dynamics (e.g., variable climate, soil differences, farming practices).


---


### Retraining Linear Regression: 

In [117]:
# Removing rows where Yield is extreme
# Defining reasonable yield range (e.g., keep only yields between 100 and 800 Bu/Acre)
filtered_data = final_merged[(final_merged['Yield'] >= 100) & (final_merged['Yield'] <= 800)].copy()

# Redefining features and target
X_filtered = filtered_data[features]
y_filtered = filtered_data['Yield']

# Train-Test split again
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_filtered, y_filtered, test_size=0.2, random_state=42
)


In [123]:
# Training linear regression again
lr_model_f = LinearRegression()
lr_model_f.fit(X_train_f, y_train_f)

# Predicting
y_pred_lr_f = lr_model_f.predict(X_test_f)


In [127]:
# Evaluating
rmse_lr_f = np.sqrt(mean_squared_error(y_test_f, y_pred_lr_f))
mae_lr_f = mean_absolute_error(y_test_f, y_pred_lr_f)
r2_lr_f = r2_score(y_test_f, y_pred_lr_f)

In [129]:
print(f"Filtered Linear Regression RMSE: {rmse_lr_f:.2f}")
print(f"Filtered Linear Regression MAE: {mae_lr_f:.2f}")
print(f"Filtered Linear Regression R²: {r2_lr_f:.2f}")

Filtered Linear Regression RMSE: 230.45
Filtered Linear Regression MAE: 186.29
Filtered Linear Regression R²: -0.21


After filtering out extreme yield outliers (only keeping records with wheat yields between 100–800 Bu/Acre),  
the Linear Regression model was retrained and evaluated.

---

###  New Model Performance:

| Metric | Value |
|:-------|:------|
| RMSE (Root Mean Squared Error) | 230.45 |
| MAE (Mean Absolute Error) | 186.29 |
| R² Score (Coefficient of Determination) | -0.21 |

---

**Comparison vs First Attempt:**

| Metric | First Attempt | After Filtering | Improvement? |
|:-------|:--------------|:----------------|:------------|
| RMSE | 344.88 | 230.45 |  Lower RMSE (better) |
| MAE | 297.35 | 186.29 |  Lower MAE (better) |
| R² | -0.78 | -0.21 |  Closer to 0 (better fit) |

---

**Key Observations:**

- **RMSE Improvement:**  
  RMSE dropped significantly from 344.88 to 230.45, meaning **average prediction errors were reduced** by over 110 bushels per acre.
  
- **MAE Improvement:**  
  MAE also dropped substantially, showing that predictions became **consistently closer to actual values**.

- **R² Score Improvement:**  
  Although still negative (-0.21), the R² score improved significantly compared to the first attempt (-0.78).  
  This suggests that the model is **closer to explaining some variance** in the data, although it still struggles to fully capture complex yield patterns.

---

**Conclusion:**

- Removing extreme outliers from the training data **meaningfully improved model performance**.
- Even a simple linear model benefitted from **cleaner, less noisy data**.
- However, linear models still **cannot capture all the non-linear complexities** of agricultural systems — hence further modeling improvements (Decision Trees, Random Forests) remain justified.

Overall, this updated Linear Regression model sets a **stronger and more fair baseline** for Sprint 2 evaluation.

---


## Baseline Model 2: Decision Tree Regressor

**Why Decision Tree Regressor?**

After observing poor performance from the Linear Regression model, it is important to try a model that can handle **non-linear relationships**, **feature interactions**, and **non-additive patterns**.

A Decision Tree Regressor:

- Automatically splits data into different regions based on feature thresholds
- Captures non-linearities without requiring feature scaling or transformations
- Can handle noisy and complex datasets better than linear models

Although Decision Trees can overfit easily, they are excellent for setting a stronger baseline compared to simple linear methods.

---

In [95]:
# Initialize model
dt_model = DecisionTreeRegressor(random_state=42)

# Train on training data
dt_model.fit(X_train, y_train)

# Predict on test data
y_pred_dt = dt_model.predict(X_test)


In [97]:
# Calculateing metrics
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
mae_dt = mean_absolute_error(y_test, y_pred_dt)
r2_dt = r2_score(y_test, y_pred_dt)

In [99]:
# Results
print(f"Decision Tree RMSE: {rmse_dt:.2f}")
print(f"Decision Tree MAE: {mae_dt:.2f}")
print(f"Decision Tree R²: {r2_dt:.2f}")

Decision Tree RMSE: 403.60
Decision Tree MAE: 329.10
Decision Tree R²: -1.43


After training and testing the Decision Tree Regressor, the following performance metrics were obtained:

| Metric | Value |
|:-------|:------|
| RMSE (Root Mean Squared Error) | 403.60 |
| MAE (Mean Absolute Error) | 329.10 |
| R² Score (Coefficient of Determination) | -1.43 |

---

**a. Root Mean Squared Error (RMSE)**

- The RMSE of 403.60 bushels per acre is **higher** than the Linear Regression RMSE (344.88).
- This suggests that the Decision Tree model **made even larger prediction errors** on average compared to the linear model.
- A very high RMSE implies poor generalization ability.

---

**b. Mean Absolute Error (MAE)**

- The MAE of 329.10 also exceeds the Linear Regression MAE of 297.35.
- On average, the Decision Tree predictions deviate by over 300 bushels per acre from the actual yields — a **very large error** in the context of typical wheat yields.

---

**c. R² Score (Coefficient of Determination)**

- A **negative R² score of -1.43** indicates **extremely poor model fit**.
- This value is even worse than that of Linear Regression (-0.78).
- A negative R² means the model is performing **worse than simply predicting the mean** of the target variable.

---

**Conclusion:**

- The Decision Tree Regressor **overfitted** the training data but failed to generalize to the test data.
- This is a classic weakness of Decision Trees when **no pruning or depth control** is applied — they memorize training data patterns that don't hold up on unseen data.
- The poor performance suggests that simple Decision Trees are not appropriate for this problem without further tuning or ensembling (e.g., Random Forests).



## Baseline Model 3: Random Forest Regressor

**Why Random Forest Regressor?**

Given the poor performance of both Linear Regression and Decision Tree models,  
it is logical to now apply a **Random Forest Regressor**, which is an ensemble method.

Random Forest:

- Combines the predictions of multiple Decision Trees (bagging approach)
- Reduces overfitting by averaging multiple trees' outputs
- Captures complex non-linear relationships
- Handles noise and variability better than a single tree

Thus, Random Forest is expected to deliver a **stronger baseline performance** for wheat yield prediction,  
especially on a noisy, real-world agricultural dataset like this one.

---

In [104]:
# Initialize model
rf_model = RandomForestRegressor(random_state=42, n_estimators=100)

# Train on training data
rf_model.fit(X_train, y_train)

# Predict on test data
y_pred_rf = rf_model.predict(X_test)

In [106]:
# Calculating metrics
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)



In [108]:
# Results
print(f"Random Forest RMSE: {rmse_rf:.2f}")
print(f"Random Forest MAE: {mae_rf:.2f}")
print(f"Random Forest R²: {r2_rf:.2f}")

Random Forest RMSE: 344.23
Random Forest MAE: 296.07
Random Forest R²: -0.77


After training and evaluating the Random Forest Regressor model, the following performance metrics were obtained:

| Metric | Value |
|:-------|:------|
| RMSE (Root Mean Squared Error) | 344.23 |
| MAE (Mean Absolute Error) | 296.07 |
| R² Score (Coefficient of Determination) | -0.77 |

---

**a. Root Mean Squared Error (RMSE)**

- RMSE of 344.23 is almost identical to the Linear Regression RMSE (344.88).
- Despite being an ensemble model, Random Forest was **not able to reduce average prediction errors significantly**.
- This suggests that the current features still lack strong direct predictive power for wheat yield.

---

**b. Mean Absolute Error (MAE)**

- MAE of 296.07 is also very close to the Linear Regression MAE (297.35).
- On average, Random Forest predictions deviate by about 296 bushels per acre from actual yields — a **very large error** relative to typical wheat yield values.

---

**c. R² Score (Coefficient of Determination)**

- An R² score of **-0.77** is almost identical to that of Linear Regression (-0.78).
- A negative R² means the model is performing **worse than simply predicting the average wheat yield** across all samples.

---

**Conclusion:**

- Despite the robustness of Random Forests for noisy and non-linear data,  
the model **does not show significant improvement** over Linear Regression or Decision Trees in this case.
- This reinforces that **feature quality** — not just model complexity — is a major bottleneck here.
- Key issues could be:
  - Lack of strong predictive signals in current features
  - High inherent noise in agricultural yield data
  - Merging mismatches or missing critical agronomic variables (e.g., pest incidents, irrigation data)

**Next Steps:**  
Future improvement would likely require **richer, domain-specific features**, better temporal alignment, or advanced techniques like feature selection, time series modeling, or external data enrichment.

---


## Baseline Model Comparison

To summarize the baseline modeling efforts, the performance of all three models (Linear Regression, Decision Tree, Random Forest) is compared below.

---

| Model | RMSE | MAE | R² | Notes |
|:------|:----:|:---:|:--:|:------|
| Linear Regression (original) | 344.88 | 297.35 | -0.78 | High error, poor fit |
| **Linear Regression (after filtering)** | **230.45** | **186.29** | **-0.21** |  Improved baseline after outlier removal |
| Decision Tree | 403.60 | 329.10 | -1.43 | Worse than Linear Regression |
| Random Forest | 344.23 | 296.07 | -0.77 | No major improvement |

---

**Key Observations:**

- **Outlier removal meaningfully improved Linear Regression** performance, reducing both RMSE and MAE significantly.
- **Decision Tree performed worse** than both versions of Linear Regression, likely due to severe overfitting.
- **Random Forest**, while generally more robust, did **not outperform** the improved Linear Regression model in this case — confirming that **feature strength is a bigger limitation** than model complexity.

The improved Linear Regression (after filtering) is accepted as the **official baseline model** for moving forward into future modeling phases.

---


## Conclusion: Baseline Modeling

- Linear Regression, Decision Tree, and Random Forest models were trained and evaluated as baseline predictors of Kansas wheat yields.
- Initial Linear Regression performance was poor,  
but **data cleaning (outlier filtering)** improved model metrics substantially.
- More complex models (Decision Tree and Random Forest) **did not outperform** the improved linear model at this stage, primarily due to **limited feature strength and data noise**.

---

**Key Takeaways:**

- **Data quality is critical:**  
  Cleaning outliers had a larger impact on model performance than switching to more complex models.
  
- **Feature limitations are real:**  
  Current features capture only part of what affects wheat yields. Factors like pest pressures, irrigation access, or more granular weather variables are missing.

- **Baseline set, but room for growth:**  
  The improved Linear Regression model now provides a reasonable baseline for Sprint 3.  
  Future work should focus on:
  - Adding better features (e.g., more detailed climate indicators)
  - Hyperparameter tuning (especially for Random Forests)
  - Advanced modeling techniques like Gradient Boosting or Time-Series approaches

 Baseline modeling is now **comprehensively completed** — setting a strong foundation for more advanced work in Sprint 3.

---
